# 01 模仿學習（Board → Action）

用啟發式教師產生的資料集訓練 IL 模型。

> 資料產生是 CPU 密集工作，**建議在本機產生**再上傳到 Drive；Colab 端只做訓練。

In [ ]:
import json
import os

os.environ.setdefault('TETRIO_AI_DRIVE', '/content/drive/MyDrive/tetrio-ai')
DRIVE_ROOT = os.environ['TETRIO_AI_DRIVE']
DATA_ROOT = os.path.join(DRIVE_ROOT, 'datasets', 'heuristic-v1')
CKPT_ROOT = os.path.join(DRIVE_ROOT, 'checkpoints', 'il')
os.makedirs(CKPT_ROOT, exist_ok=True)
print('dataset:', DATA_ROOT)
print('checkpoints:', CKPT_ROOT)
print(json.load(open(os.path.join(DATA_ROOT, 'manifest.json'), encoding='utf-8')) if os.path.exists(DATA_ROOT) else '尚未有資料集')

In [ ]:
# （可選）若 Drive 上還沒有資料集，直接在 Colab 產生一份小的
if not os.path.exists(DATA_ROOT):
    !python -m scripts.generate_dataset --out {DATA_ROOT} --episodes 40 --workers 4 --max-pieces 200

In [ ]:
# 先用小樣本確認 loss 會下降，再放大到全量
!python -m scripts.train_il --data-root {DATA_ROOT} --epochs 3 --limit 20000 \
    --checkpoint-dir {CKPT_ROOT} --network resnet --out {CKPT_ROOT}/il_small.json

In [ ]:
# 正式訓練（全量資料，約 30 epochs + early stopping）
!python -m scripts.train_il --data-root {DATA_ROOT} --epochs 30 \
    --checkpoint-dir {CKPT_ROOT} --network resnet --out {CKPT_ROOT}/il_full.json

In [ ]:
# 驗收：top-1 ≥ 0.85、top-3 ≥ 0.97（configs/il.yaml 的 metrics 門檻）
print(json.load(open(os.path.join(CKPT_ROOT, 'il_full.json'), encoding='utf-8'))['best_top1'])